**Notebook to develop and test**

In [ ]:
import os
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
from IPython.display import Image

# Add the scripts/pypsa-de directory to the path
scripts_path = os.path.join(os.path.dirname(os.getcwd()), "scripts", "pypsa-de")
sys.path.append(scripts_path)

from flexibility_utils import tech_colors

In [ ]:
# load networks

networks = {}
run = "20250214-reworkimportban"  # "20241203-force-onwind-south"
scenario = "KN2045_Bal_v4"  # "KN2045_Elec_v4" # "KN2045_H2_v4" # "CurrentPolicies" # "KN2045_Bal_v4"

for year in np.arange(2020, 2050, 5):
    fn = f"/home/julian-geis/Documents/04_Ariadne/run_results/{run}/{scenario}/postnetworks/base_s_49_lvopt__none_{year}.nc"
    networks[year] = pypsa.Network(fn)

# System cost

In [ ]:
from pypsa.statistics import get_transmission_carriers


def get_system_costs(n, country=None):
    """
    Calculate capex and opex for entire system or specific country.

    Parameters
    ----------
    n : pypsa.Network
    country : str, optional
        Country code (e.g., "DE", "EU"). If None, returns total system costs.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ["capex", "opex"] in billion EUR, indexed by carrier
    """
    # Assign country from location or bus name if empty
    empty_country = n.buses.country == ""
    n.buses.loc[empty_country, "country"] = n.buses.loc[empty_country, "location"]
    still_empty = n.buses.country == ""
    n.buses.loc[still_empty, "country"] = n.buses.loc[still_empty].index.str[:2]

    if country is None:
        return (
            pd.DataFrame(
                {
                    "capex": n.statistics.capex(groupby="carrier", nice_names=False),
                    "opex": n.statistics.opex(groupby="carrier", nice_names=False),
                }
            )
            / 1e9
        )

    # Country-specific costs
    pypsa.options.set_option("params.statistics.drop_zero", False)
    kwargs = {"groupby": ["name", "carrier"], "nice_names": False}

    costs = pd.DataFrame(
        {"capex": n.statistics.capex(**kwargs), "opex": n.statistics.opex(**kwargs)}
    ).reset_index()

    # Map component names to countries
    def get_component_country(name):
        component_attrs = {
            "generators": "bus",
            "loads": "bus",
            "stores": "bus",
            "storage_units": "bus",
            "links": "bus0",
            "lines": "bus0",
        }
        for attr, bus_col in component_attrs.items():
            df = getattr(n, attr)
            if name in df.index:
                return n.buses.loc[df.loc[name, bus_col], "country"]
        return None

    costs["country"] = costs["name"].apply(get_component_country)

    # Get transmission carriers
    transmission_carriers = (
        get_transmission_carriers(n).get_level_values("carrier").unique()
    )

    # Find inter-country transmission touching this country
    def inter_country_mask(df):
        bus0_country = df.bus0.map(n.buses.country)
        bus1_country = df.bus1.map(n.buses.country)
        return (
            df.carrier.isin(transmission_carriers)
            & df.active
            & (bus0_country != bus1_country)
            & ((bus0_country == country) | (bus1_country == country))
        )

    inter_assets = inter_country_mask(n.lines)[lambda x: x].index.union(
        inter_country_mask(n.links)[lambda x: x].index
    )

    # Split shared transmission costs
    mask = costs["name"].isin(inter_assets)
    costs.loc[mask, ["capex", "opex"]] /= 2

    # Filter to country and convert to billion EUR
    return (
        costs[costs["country"] == country].groupby("carrier")[["capex", "opex"]].sum()
        / 1e9
    )

In [ ]:
def get_system_costs(n, country=None):
    """
    Calculate capex and opex for entire system or specific country.

    Parameters
    ----------
    n : pypsa.Network
    country : str, optional
        Country code (e.g., "DE"). If None, returns total system costs.
        Note: "EU" buses are commodity buses and costs are assigned to real countries.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ["capex", "opex"] in billion EUR, indexed by carrier
    """
    # Assign country from location or bus name if empty
    empty_country = n.buses.country == ""
    n.buses.loc[empty_country, "country"] = n.buses.loc[empty_country, "location"]
    still_empty = n.buses.country == ""
    n.buses.loc[still_empty, "country"] = n.buses.loc[still_empty].index.str[:2]

    if country is None:
        return (
            pd.DataFrame(
                {
                    "capex": n.statistics.capex(groupby="carrier", nice_names=False),
                    "opex": n.statistics.opex(groupby="carrier", nice_names=False),
                }
            )
            / 1e9
        )

    # Country-specific costs
    pypsa.options.set_option("params.statistics.drop_zero", False)
    kwargs = {"groupby": ["name", "carrier"], "nice_names": False}

    costs = pd.DataFrame(
        {"capex": n.statistics.capex(**kwargs), "opex": n.statistics.opex(**kwargs)}
    ).reset_index()

    # Map components to country based on buses
    def get_component_country(name):
        # Single-bus components
        for attr in ["generators", "loads", "stores", "storage_units"]:
            df = getattr(n, attr)
            if name in df.index:
                return n.buses.loc[df.loc[name, "bus"], "country"]

        # Two-bus components: assign based on real country (not EU)
        for attr in ["links", "lines"]:
            df = getattr(n, attr)
            if name in df.index:
                bus0_country = n.buses.loc[df.loc[name, "bus0"], "country"]
                bus1_country = n.buses.loc[df.loc[name, "bus1"], "country"]

                # If one side is EU, assign to the other side
                if bus0_country == "EU":
                    return bus1_country
                elif bus1_country == "EU":
                    return bus0_country
                else:
                    # Both are real countries - return tuple for inter-country check
                    return (bus0_country, bus1_country)
        return None

    costs["country_info"] = costs["name"].apply(get_component_country)

    # Get transmission carriers
    transmission_carriers = (
        get_transmission_carriers(n).get_level_values("carrier").unique()
    )

    # Find inter-country transmission touching this country (excluding EU buses)
    def inter_country_mask(df):
        bus0_country = df.bus0.map(n.buses.country)
        bus1_country = df.bus1.map(n.buses.country)
        return (
            df.carrier.isin(transmission_carriers)
            & df.active
            & (bus0_country != bus1_country)
            & (bus0_country != "EU")
            & (bus1_country != "EU")  # Exclude EU buses
            & ((bus0_country == country) | (bus1_country == country))
        )

    inter_assets = inter_country_mask(n.lines)[lambda x: x].index.union(
        inter_country_mask(n.links)[lambda x: x].index
    )

    # Split shared transmission costs
    mask = costs["name"].isin(inter_assets)
    costs.loc[mask, ["capex", "opex"]] /= 2

    # Filter to country
    def belongs_to_country(country_info):
        if country_info is None:
            return False
        if isinstance(country_info, tuple):
            # Inter-country transmission: belongs if either bus is in country
            return country in country_info
        else:
            # Direct assignment or single-bus component
            return country_info == country

    country_mask = costs["country_info"].apply(belongs_to_country)

    return costs[country_mask].groupby("carrier")[["capex", "opex"]].sum() / 1e9

In [ ]:
# All individual countries + EU
n = networks[2045].copy()

sum = 0
countries = n.buses.country[n.buses.country != ""].unique().tolist() + ["EU"]
res_cl = {}

for c in countries:
    res_cl[c] = get_system_costs(n, country=c)
    print(f"{c}: {res_cl[c].sum().sum():.2f} bn EUR")
    sum += res_cl[c].sum().sum()
print(f"Sum of countries: {sum:.2f} bn EUR")
print()

# Total
total = get_system_costs(n)
print(f"Total: {total.sum().sum():.2f} bn EUR")

In [ ]:
# system cost DE (according to exporter)
region = "DE"


def get_tsc(n, country):
    pypsa.options.set_option("params.statistics.drop_zero", False)
    capex = n.statistics.capex(
        groupby=pypsa.statistics.groupers["name", "carrier"], nice_names=False
    )

    opex = n.statistics.opex(
        groupby=pypsa.statistics.groupers["name", "carrier"], nice_names=False
    )

    # filter inter country transmission lines and links
    inter_country_lines = n.lines.bus0.map(n.buses.country) != n.lines.bus1.map(
        n.buses.country
    )
    inter_country_links = n.links.bus0.map(n.buses.country) != n.links.bus1.map(
        n.buses.country
    )
    #
    transmission_carriers = get_transmission_carriers(n).get_level_values("carrier")
    transmission_lines = n.lines.carrier.isin(transmission_carriers) & n.lines.active
    transmission_links = n.links.carrier.isin(transmission_carriers) & n.links.active
    #
    country_transmission_lines = (
        (n.lines.bus0.str.contains(country)) & ~(n.lines.bus1.str.contains(country))
    ) | (~(n.lines.bus0.str.contains(country)) & (n.lines.bus1.str.contains(country)))
    country_tranmission_links = (
        (n.links.bus0.str.contains(country)) & ~(n.links.bus1.str.contains(country))
    ) | (~(n.links.bus0.str.contains(country)) & (n.links.bus1.str.contains(country)))
    #
    inter_country_transmission_lines = (
        inter_country_lines & transmission_lines & country_transmission_lines
    )
    inter_country_transmission_links = (
        inter_country_links & transmission_links & country_tranmission_links
    )
    inter_country_transmission_lines_i = inter_country_transmission_lines[
        inter_country_transmission_lines
    ].index
    inter_country_transmission_links_i = inter_country_transmission_links[
        inter_country_transmission_links
    ].index
    inter_country_transmission_i = inter_country_transmission_lines_i.union(
        inter_country_transmission_links_i
    )

    #
    tsc = pd.concat([capex, opex], axis=1, keys=["capex", "opex"])
    tsc = tsc.reset_index().set_index("name")
    tsc.loc[inter_country_transmission_i, ["capex", "opex"]] = (
        tsc.loc[inter_country_transmission_i, ["capex", "opex"]] / 2
    )
    tsc.rename(
        index={index: index + " " + country for index in inter_country_transmission_i},
        inplace=True,
    )
    # rename inter region links and lines
    to_rename_links = n.links[
        (n.links.bus0.str.contains(region))
        & (n.links.bus1.str.contains(region))
        & ~(n.links.index.str.contains(region))
    ].index
    to_rename_lines = n.lines[
        (n.lines.bus0.str.contains(region))
        & (n.lines.bus1.str.contains(region))
        & ~(n.lines.index.str.contains(region))
    ].index
    tsc.rename(
        index={index: index + " " + region for index in to_rename_links},
        inplace=True,
    )
    tsc.rename(
        index={index: index + " " + region for index in to_rename_lines},
        inplace=True,
    )

    tsc = (
        tsc.filter(like=country, axis=0)
        .drop("component", axis=1)
        .groupby("carrier")
        .sum()
    )

    return tsc

In [ ]:
n = networks[2045].copy()
tsc_de = get_tsc(n, "DE")

In [ ]:
# All individual countries + EU
n = networks[2045].copy()

sum = 0
countries = n.buses.country[n.buses.country != ""].unique().tolist() + ["EU"]
res = {}

for c in countries:
    res[c] = get_tsc(n, country=c)
    print(f"{c}: {res[c].sum().sum() / 1e9:.2f} bn EUR")
    sum += res[c].sum().sum()
print(f"Sum of countries: {sum / 1e9:.2f} bn EUR")
print()

In [ ]:
# AT: 13.47 bn EUR
# BE: 14.06 bn EUR
# CH: 13.31 bn EUR
# CZ: 11.80 bn EUR
# DE: 151.37 bn EUR
# DK: 14.44 bn EUR
# ES: 75.47 bn EUR
# FR: 122.23 bn EUR
# GB: 90.56 bn EUR
# IT: 92.06 bn EUR
# LU: 1.45 bn EUR
# NL: 37.11 bn EUR
# NO: 13.91 bn EUR
# PL: 65.98 bn EUR
# SE: 14.56 bn EUR
# EU: 77.11 bn EUR
# Sum of countries: 808.89 bn EUR

# Total: 820.47 bn EUR

In [ ]:
res["CH"].sum() / 1e9

In [ ]:
res_cl["CH"].sum()

In [ ]:
df = res_cl["CH"]["capex"]
df.sort_values(ascending=False)[:30] * 1e3

In [ ]:
df = res["CH"]["capex"]
(df[df > 1e6] / 1e6).sort_values(ascending=False)

In [ ]:
n.links[
    (n.links.carrier == "urban central gas CHP") & (n.links.bus1.str.contains("CH"))
][["p_nom_opt"]]

In [ ]:
n.statistics.capex(groupby=["name", "carrier"], nice_names=False)["Link"][
    ["CH0 0 urban central gas CHP-2045", "CH0 0 urban central gas CHP-2010"]
].sum() / 1e6

In [ ]:
n.statistics.capex(groupby=["bus", "carrier"], nice_names=False)

# Industry DSM

In [ ]:
def add_industry_dsm(n, dsm_config):
    """
    Add demand-side management (DSM) for industrial loads in Germany.

    Creates distributed DSM capacity across industrial load buses, weighted by
    their load. DSM is implemented as a debt-tracking system where load can be
    reduced (creating debt) and must be compensated later.

    Parameters
    ----------
    n : pypsa.Network
        The energy system network.
    dsm_config : dict
        Configuration dictionary with keys under 'industry_dsm_de':
        - shift_capacity: float, total DSM capacity in GW
        - holding_hours: float, maximum hours debt can be held
        - ramp_down_cost: float, marginal cost of reducing load (€/MWh)
    """

    # logger.info("Adding demand-side management (DSM) for industrial loads in Germany.")

    # Filter industrial loads in Germany
    industrial_loads = n.loads[
        (n.loads.carrier == "industry electricity")
        & (n.loads.bus.str.contains("DE", na=False))
    ]

    # if industrial_loads.empty:
    #     logger.warning("No industrial loads found in Germany. Skipping DSM addition.")
    #     return

    # Calculate weights based on average load at each bus
    load_by_bus = {}

    for load_name in industrial_loads.index:
        bus = n.loads.at[load_name, "bus"]

        # Get average load for weighting
        if load_name in n.loads_t.p_set.columns:
            avg_load = n.loads_t.p_set[load_name].mean()
        elif hasattr(n.loads.at[load_name, "p_set"], "__iter__"):
            avg_load = n.loads.at[load_name, "p_set"].mean()
        else:
            avg_load = n.loads.at[load_name, "p_set"]

        if bus in load_by_bus:
            load_by_bus[bus] += avg_load
        else:
            load_by_bus[bus] = avg_load

    total_load = sum(load_by_bus.values())

    # if total_load == 0:
    #     logger.warning("Total industrial load in Germany is zero. Skipping DSM addition.")
    #     return

    # Extract config parameters
    total_shift_capacity = dsm_config["shift_capacity"] * 1e3  # Convert GW to MW
    holding_hours = dsm_config["holding_hours"]
    ramp_down_cost = dsm_config.get("ramp_down_cost", 0)
    compensate_cost = dsm_config.get("compensate_cost", 0)

    # logger.info(f"Total DSM capacity: {dsm_config['shift_capacity']} GW, "
    #             f"Holding time: {holding_hours} hours, "
    #             f"distributed across {len(load_by_bus)} buses")

    # Define carriers
    n.add("Carrier", "industry DSM")
    n.add("Carrier", "industry DSM ramp down")
    n.add("Carrier", "industry DSM compensate")

    # Add DSM components for each bus
    for bus, load in load_by_bus.items():
        # Calculate this bus's share of DSM capacity based on load weight
        weight = load / total_load
        bus_shift_capacity = total_shift_capacity * weight  # MW
        bus_debt_capacity = bus_shift_capacity * holding_hours  # MWh

        # logger.debug(f"Bus {bus}: {bus_shift_capacity:.1f} MW DSM ({weight*100:.1f}%)")

        # Create bus for debt tracking
        dsm_bus_name = f"{bus} DSM debt bus"
        n.add("Bus", dsm_bus_name, carrier="DSM")

        # Store to track production debt
        n.add(
            "Store",
            f"{bus}_DSM_debt",
            bus=dsm_bus_name,
            e_nom=bus_debt_capacity,
            e_initial=0,
            standing_loss=0,
            capital_cost=0,
            carrier="industry DSM",
        )

        # Link: Ramp DOWN = reduce load, charges the debt store
        n.add(
            "Link",
            f"{bus} DSM ramp down",
            bus0=bus,  # Takes from grid
            bus1=dsm_bus_name,  # Charges debt
            p_nom=bus_shift_capacity,
            efficiency=1.0,
            marginal_cost=ramp_down_cost,
            capital_cost=0,
            carrier="industry DSM ramp down",
        )

        # Link: Compensate = increase load later to catch up, discharges debt
        n.add(
            "Link",
            f"{bus} DSM compensate",
            bus0=dsm_bus_name,  # Discharges debt
            bus1=bus,  # Adds load to grid
            p_nom=bus_shift_capacity,
            efficiency=1.0,
            marginal_cost=compensate_cost,
            capital_cost=0,
            carrier="industry DSM compensate",
        )

    print(
        f"Added DSM components to {len(load_by_bus)} buses. "
        f"Total capacity: {total_shift_capacity / 1e3:.2f} GW, "
        f"Total storage: {total_shift_capacity * holding_hours / 1e3:.2f} GWh"
    )

In [ ]:
n.stores.index[n.stores.carrier == "industry_DSM"]

In [ ]:
# industry dsm
config = {
    "shift_capacity": 10,  # GW
    "holding_hours": 4,
    "compensate_hours": 24,
    "ramp_down_cost": 0,
    "compensate_cost": 0,
}

In [ ]:
add_industry_dsm(n, config)

In [ ]:
# Find all DSM debt stores
dsm_stores = n.stores.index[n.stores.carrier == "industry DSM"]


compensate_hours = 24
# Convert snapshots to hours from start
start_time = n.snapshots[0]
snapshot_hours = [(snap - start_time).total_seconds() / 3600 for snap in n.snapshots]

# Find snapshots that are at multiples of compensate_hours
# Allow some tolerance for floating point comparison
tolerance = 0.1  # hours
cycle_snapshots = []

for i, hours in enumerate(snapshot_hours):
    # Check if this snapshot is at a multiple of compensate_hours
    remainder = hours % compensate_hours
    # Check both remainder near 0 or near compensate_hours (wrapping)
    if remainder < tolerance or (compensate_hours - remainder) < tolerance:
        cycle_snapshots.append(n.snapshots[i])

# Always include the last snapshot to ensure debt is cleared at end
if n.snapshots[-1] not in cycle_snapshots:
    cycle_snapshots.append(n.snapshots[-1])

if not cycle_snapshots:
    print(
        f"No snapshots found at {compensate_hours}h intervals. "
        f"Temporal resolution may be too coarse. Adding constraint only at last snapshot."
    )
    cycle_snapshots = [n.snapshots[-1]]

# Calculate average snapshot duration for info
if len(n.snapshots) > 1:
    avg_duration = sum(
        (n.snapshots[i + 1] - n.snapshots[i]).total_seconds() / 3600
        for i in range(len(n.snapshots) - 1)
    ) / (len(n.snapshots) - 1)
    print(f"Average snapshot duration: {avg_duration:.2f} hours")

print(
    f"DSM debt must be zero at {len(cycle_snapshots)} snapshots "
    f"(approximately every {compensate_hours} hours)"
)

# Add constraints for each store at each cycle point
constraint_count = 0
for store in dsm_stores:
    for snapshot in cycle_snapshots:
        cname = f"DSM_cycling-{store}-{snapshot}"

        # Store state of charge must be zero at this snapshot
        lhs = n.model["Store-e"].loc[snapshot, store]

        n.model.add_constraints(lhs == 0, name=cname)
        constraint_count += 1

# not adding to network as the shadow prices are not needed

print(
    f"Added {constraint_count} DSM cycling constraints across {len(dsm_stores)} stores"
)

In [ ]:
n.stores.index[n.stores.carrier == "DSM"]